In [ ]:
"""
Step 1: Extract PDFs with Docling + Surya OCR.
Processes one file at a time with memory cleanup.
Skips already-processed files (resume-safe).
"""
import os
import gc
import json
import torch
from pathlib import Path
from docling.datamodel.base_models import InputFormat
from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
from docling.datamodel.pipeline_options import (
    ThreadedPdfPipelineOptions,
    TableStructureOptions,
)
from docling.datamodel.settings import settings
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling_surya import SuryaOcrOptions

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PDF_DIR = "pdfs"
MD_DIR = "work/md_output"
OBJ_DIR = "work/doc_objects"

os.makedirs(MD_DIR, exist_ok=True)
os.makedirs(OBJ_DIR, exist_ok=True)

settings.debug.profile_pipeline_timings = True

pipeline_options = ThreadedPdfPipelineOptions(
    do_ocr=True,
    ocr_model="suryaocr",
    allow_external_plugins=True,
    ocr_options=SuryaOcrOptions(lang=["en"]),
    do_table_structure=True,
    table_structure_options=TableStructureOptions(do_cell_matching=True),
    generate_page_images=True,
    generate_picture_images=True,
    images_scale=2.0,
    accelerator_options=AcceleratorOptions(
        num_threads=8,
        device=AcceleratorDevice.CUDA,
    ),
    ocr_batch_size=16,
    layout_batch_size=16,
    table_batch_size=2,
)

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options),
    }
)

# Sort by file size — small files first, big files last
pdf_files = sorted(Path(PDF_DIR).glob("*.pdf"), key=lambda p: p.stat().st_size)
print(f"Found {len(pdf_files)} PDFs (sorted smallest → largest)")

results = {}
# Load existing results if resuming
results_path = Path(MD_DIR) / "_summary.json"
if results_path.exists():
    with open(results_path) as f:
        results = json.load(f)
    print(f"Resuming: {sum(1 for v in results.values() if v.get('status') == 'success')} already done")

for i, pdf_path in enumerate(pdf_files):
    # Skip if already processed
    if pdf_path.name in results and results[pdf_path.name].get("status") == "success":
        print(f"[{i+1}/{len(pdf_files)}] SKIP {pdf_path.name} (already done)")
        continue

    size_mb = pdf_path.stat().st_size / 1e6
    print(f"[{i+1}/{len(pdf_files)}] Processing {pdf_path.name} ({size_mb:.1f} MB)...")

    try:
        result = converter.convert(str(pdf_path))
        doc = result.document

        # Save markdown
        md_text = doc.export_to_markdown()
        md_path = Path(MD_DIR) / f"{pdf_path.stem}.md"
        md_path.write_text(md_text, encoding="utf-8")

        # Save structured document object as JSON
        obj_path = Path(OBJ_DIR) / f"{pdf_path.stem}.json"
        doc.save_as_json(obj_path)

        results[pdf_path.name] = {
            "status": "success",
            "chars": len(md_text),
            "obj_size": obj_path.stat().st_size,
        }
        print(f"  OK: {len(md_text):,} chars, obj: {obj_path.stat().st_size:,} bytes")

        # Cleanup document objects
        del doc, md_text

    except Exception as e:
        print(f"  ERROR: {e}")
        results[pdf_path.name] = {"status": "error", "error": str(e)}

    # Memory cleanup after every file
    try:
        del result
    except NameError:
        pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        mem_used = torch.cuda.memory_allocated() / 1e9
        mem_total = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"  GPU memory: {mem_used:.1f}/{mem_total:.1f} GB")

    # Save progress after every file (resume-safe)
    with open(results_path, "w") as f:
        json.dump(results, f, indent=2)

print(f"\nDone. {sum(1 for v in results.values() if v.get('status')=='success')}/{len(results)} succeeded.")

In [ ]:
import os
import gc
import json
import torch
from pathlib import Path
from docling.datamodel.base_models import InputFormat
from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
from docling.datamodel.pipeline_options import (
    ThreadedPdfPipelineOptions,
    TableStructureOptions,
)
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling_surya import SuryaOcrOptions

PDF_PATH = "pdfs/66156-4274314.pdf"
MD_DIR = "work/md_output"
OBJ_DIR = "work/doc_objects"

# Minimal batch sizes for 441-page monster
pipeline_options = ThreadedPdfPipelineOptions(
    do_ocr=True,
    ocr_model="suryaocr",
    allow_external_plugins=True,
    ocr_options=SuryaOcrOptions(lang=["en"]),
    do_table_structure=True,
    table_structure_options=TableStructureOptions(do_cell_matching=True),
    generate_page_images=True,
    generate_picture_images=True,
    images_scale=1.5,        # Reduced from 2.0 — less VRAM per page
    accelerator_options=AcceleratorOptions(
        num_threads=8,
        device=AcceleratorDevice.CUDA,
    ),
    ocr_batch_size=4,        # Minimal
    layout_batch_size=4,     # Minimal
    table_batch_size=1,      # Minimal
)

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options),
    }
)

gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory before: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
print(f"Processing {PDF_PATH}...")

result = converter.convert(PDF_PATH)
doc = result.document

md_text = doc.export_to_markdown()
Path(MD_DIR, "66156-4274314.md").write_text(md_text, encoding="utf-8")

doc.save_as_json(Path(OBJ_DIR, "66156-4274314.json"))

print(f"OK: {len(md_text):,} chars")

# Update summary
summary_path = Path(MD_DIR) / "_summary.json"
with open(summary_path) as f:
    results = json.load(f)
results["66156-4274314.pdf"] = {
    "status": "success",
    "chars": len(md_text),
    "obj_size": Path(OBJ_DIR, "66156-4274314.json").stat().st_size,
}
with open(summary_path, "w") as f:
    json.dump(results, f, indent=2)

print("Done. 70/70 complete.")

In [ ]:
from pathlib import Path
from docling_core.types.doc.document import DoclingDocument
import json

test_files = [
    ("56061-4538520.json", "Simple"),
    ("283789-5060638.json", "Formulary"),
    ("66156-4274314.json", "Flowchart"),
]

for fname, label in test_files:
    p = Path(f"work/doc_objects/{fname}")
    print(f"\n{label}: {fname} ({p.stat().st_size / 1e6:.1f} MB)")

    doc = DoclingDocument.load_from_json(p)
    items = list(doc.iterate_items())
    print(f"  Items: {len(items)}")
    print(f"  Tables: {len(doc.tables)}")
    print(f"  Texts: {len(doc.texts)}")

    # Verify PsO content is findable
    pso_found = False
    for item, level in items:
        text = getattr(item, 'text', '')
        if 'plaque psoriasis' in text.lower() or 'pso' in text.lower():
            pso_found = True
            break
    print(f"  PsO content found: {pso_found}")

print("\nAll objects loadable. Ready to build reducer.")

In [ ]:
from pathlib import Path
from docling_core.types.doc.document import DoclingDocument

doc = DoclingDocument.load_from_json(Path("work/doc_objects/66156-4274314.json"))

# Find tables containing CIMZIA, psoriasis, or criteria routing
print("=== TABLES WITH CIMZIA/PsO/ROUTING ===\n")
for i, table in enumerate(doc.tables):
    try:
        df = table.export_to_dataframe(doc)
    except:
        try:
            df = table.export_to_dataframe()
        except:
            continue
    
    df_str = df.to_string().lower()
    
    has_cimzia = 'cimzia' in df_str or 'certolizumab' in df_str
    has_pso = 'plaque psoriasis' in df_str or 'psoriasis' in df_str
    has_routing = 'go to #' in df_str or 'go to #' in df_str
    
    if has_cimzia or has_pso or has_routing:
        tags = []
        if has_cimzia: tags.append("CIMZIA")
        if has_pso: tags.append("PsO")
        if has_routing: tags.append("ROUTING")
        
        print(f"--- Table {i} [{', '.join(tags)}] Shape: {df.shape} ---")
        print(f"Columns: {list(df.columns)}")
        # Print rows that match
        for r in range(len(df)):
            row_str = ' '.join(str(v) for v in df.iloc[r].tolist()).lower()
            if any(kw in row_str for kw in ['cimzia', 'certolizumab', 'psoriasis', 'pso', 'go to #10', 'go to #11', '#10', '#11']):
                print(f"  Row {r}: {[str(v)[:60] for v in df.iloc[r].tolist()]}")
        print()

print("=== Done ===")

In [ ]:
from pathlib import Path
from docling_core.types.doc.document import DoclingDocument

doc = DoclingDocument.load_from_json(Path("work/doc_objects/66156-4274314.json"))

items = list(doc.iterate_items())

# Find tables 440-455 and their surrounding context
table_count = 0
for i, (item, level) in enumerate(items):
    item_type = type(item).__name__
    if item_type == 'TableItem':
        table_count += 1
        if 440 <= table_count <= 455:
            text = getattr(item, 'text', '')[:80]
            print(f"  [{i}] L{level} TABLE #{table_count}: {text}")
    elif item_type == 'SectionHeaderItem' and 3400 <= i <= 3700:
        text = getattr(item, 'text', '')[:80]
        print(f"  [{i}] L{level} HEADER: {text}")

In [ ]:
from pathlib import Path
from docling_core.types.doc.document import DoclingDocument

doc = DoclingDocument.load_from_json(Path("work/doc_objects/66156-4274314.json"))

items = list(doc.iterate_items())

# Show everything from item 3280 to 3340
for i, (item, level) in enumerate(items):
    if 3280 <= i <= 3340:
        item_type = type(item).__name__
        text = getattr(item, 'text', '')[:80]
        print(f"  [{i}] L{level} {item_type}: {text}")